Script para definir qual será o método final usado para o restante dos testes para inclusão no artigo. A seleção funcionará da seguinte forma:

- Para *BRKGAs* (puro e com mutação personalizada)
1. comparação entre mesmo método com diferents valores de stag (fixar o melhor valor de stag)
2. comparação entre métodos diferentes (com o parâmetro de stag que melhora a performance de cada um já definido)
3. o método que "vence" no passo 2 será o escolhido para a apresentação de resultados no artigo

- Para *Genéticos* (simple_mean e binomial) 
1. comparação entre mesmo método com mesma probabilidade e diferentes valores de stag (primeiro, fixamos o valor de stag para cada possível probabilidade testada)
2. comparação entre mesmo método com diferentes probabilidades (com o valor de stag já fixado)
    * a ideia é que apoós esse passo tenhamos definido qual é a melhor combinação de parâmetros para cada método, e partir disso compararemos as melhores versões para decidir qual será o método final utilizado
3. comparação entre métodos diferentes (com valores de probabilidade e stag que maximizam cada performance já definidos)
4. o método que "vence" no passo 3 será o escolhido para apresentação de resultados no 

* as análises serão separadas por classes de grafos - primeiro grafos bipartidos e depois grafos simples

**PARTE 1 - TESTES DE PERFORMANCE PARA GRAFOS SIMPLES**

**PARTE 1.1 - TESTES COM BRKGAs**

In [2]:
# 1° passo: tratar os arquivos que serão usados nas análises e carregá-los

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy

# append os arquivos com n = 500 separado ao restante do arquivo com os mesmos parâmetros
# append para BRKGAmutation com stag = 50
df_brkgamut_base_50 = pd.read_csv('results_BRKGAmutation_stag50_progresso.csv')
df_brkgamut_n500_50 = pd.read_csv('results_BRKGAmutation_n500_stag50_progresso.csv')

df_brkgamut_full = pd.concat([df_brkgamut_base_50, df_brkgamut_n500_50], ignore_index=True)
df_brkgamut_full.to_csv('results_BRKGAmutation_combined_stag50.csv', index=False)

# append para BRKGAmutation com stag = 20
df_brkgamut_base_20 = pd.read_csv('results_BRKGAmutation_stag20_progresso.csv')
df_brkgamut_n500_20 = pd.read_csv('results_BRKGAmutation_n500_stag20_progresso.csv')

df_brkgamut_full_20 = pd.concat([df_brkgamut_base_20, df_brkgamut_n500_20], ignore_index=True)
df_brkgamut_full_20.to_csv('results_BRKGAmutation_combined_stag20.csv', index=False)

# append para BRKGA puro com stag = 50
df_brkga_base_50 = pd.read_csv('results_BRKGA_stag50_progresso.csv')
df_brkga_n500_50 = pd.read_csv('results_BRKGA_n500_stag50_progresso.csv')

df_brkga_full_50 = pd.concat([df_brkga_base_50, df_brkga_n500_50], ignore_index=True)
df_brkga_full_50.to_csv('results_BRKGA_combined_stag50.csv', index=False)

# append para BRKGA puro com stag = 20
df_brkga_base_20 = pd.read_csv('results_BRKGA_stag20_progresso.csv')
df_brkga_n500_20 = pd.read_csv('results_BRKGA_n500_stag20_progresso.csv')

df_brkga_full_20 = pd.concat([df_brkga_base_20, df_brkga_n500_20], ignore_index=True)
df_brkga_full_20.to_csv('results_BRKGA_combined_stag20.csv', index=False)

In [3]:
# helper function para comparações absolutas entre os métodos
def get_detailed_stats(row):
    # 1. NaN Protection: If any of the 4 key values are missing, it's Incomplete
    required_cols = ['mean_chi_Stag20', 'mean_chi_Stag50', 'mean_time_Stag20', 'mean_time_Stag50']
    if any(pd.isna(row[col]) for col in required_cols):
        return 'Incomplete', 'Incomplete', 'Incomplete'
    
    # --- A. STRICTLY BY CHI (Quality Only) ---
    if row['mean_chi_Stag20'] < row['mean_chi_Stag50']:
        chi_winner = 'Stag 20'
    elif row['mean_chi_Stag50'] < row['mean_chi_Stag20']:
        chi_winner = 'Stag 50'
    else:
        chi_winner = 'Tied'

    # --- B. STRICTLY BY TIME (Speed Only) ---
    if row['mean_time_Stag20'] < row['mean_time_Stag50']:
        time_winner = 'Stag 20'
    elif row['mean_time_Stag50'] < row['mean_time_Stag20']:
        time_winner = 'Stag 50'
    else:
        time_winner = 'Tied'

    # --- C. ABSOLUTE WINNER (Primary: Chi, Secondary: Time) ---
    if chi_winner != 'Tied':
        abs_winner = chi_winner
    else:
        # If the coloring quality is exactly the same, the faster one wins the "Absolute" title
        abs_winner = time_winner 

    return chi_winner, time_winner, abs_winner

In [4]:
# 1B. determinar qual é o melhor do mesmo método e diferentes valores de stag
# combined method (analyze both methods at a time with different stags)
import pandas as pd

# 1. Define a function to process any BRKGA comparison
def analyze_brkga_performance(files_dict, title_name):
    all_results = []
    cols_to_keep = ['instancia', 'mean_chi', 'mean_time']
    
    for label, file_path in files_dict.items():
        df = pd.read_csv(file_path)
        df_filtered = df[[c for c in cols_to_keep if c in df.columns]].copy()
        df_filtered['parameter_id'] = label
        all_results.append(df_filtered)
    
    df_combined = pd.concat(all_results, ignore_index=True)
    
    # Pivot to side-by-side
    df_pivot = df_combined.pivot(index='instancia', columns='parameter_id', values=['mean_chi', 'mean_time'])
    df_pivot.columns = [f'{col}_{val}'.replace(' ', '') for col, val in df_pivot.columns]
    df_pivot = df_pivot.reset_index()

    # Apply the Winner Logic
    df_pivot[['chi_winner', 'time_winner', 'abs_winner']] = df_pivot.apply(
        lambda r: pd.Series(get_detailed_stats(r)), axis=1
    )

    valid = df_pivot[df_pivot['chi_winner'] != 'Incomplete']

    # PRINT SUMMARY
    print("\n" + "="*60)
    print(f"{title_name:^60}")
    print("="*60)
    print(f"STRICTLY BY CHI:\n{valid['chi_winner'].value_counts().to_string()}\n")
    print(f"STRICTLY BY TIME:\n{valid['time_winner'].value_counts().to_string()}\n")
    print(f"ABSOLUTE WINNER:\n{valid['abs_winner'].value_counts().to_string()}")
    print("="*60 + "\n")

    return df_pivot # This is the key to fixing your error

# --- 3. EXECUTION ---

# RUN FOR PURE BRKGA
pure_files = {
    "Stag 20": "results_BRKGA_combined_stag20.csv",
    "Stag 50": "results_BRKGA_combined_stag50.csv"
}
# We store the returned pivot table in 'df_pure_pivot'
df_pure_pivot = analyze_brkga_performance(pure_files, "RESULTS: PURE BRKGA")

# RUN FOR MUTATION BRKGA
mutation_files = {
    "Stag 20": "results_BRKGAmutation_combined_stag20.csv",
    "Stag 50": "results_BRKGAmutation_combined_stag50.csv"
}
# We store the returned pivot table in 'df_mut_pivot'
df_mut_pivot = analyze_brkga_performance(mutation_files, "RESULTS: BRKGA WITH MUTATION")


# --- 4. QUICK CHECKS (RUNNING RIGHT AFTER) ---

# Define a quick helper to print the Stag 50 margins without repeating code
def check_stag50_margins(df_to_check, analysis_type):
    stag50_chi_wins = df_to_check[df_to_check['chi_winner'] == 'Stag 50'].copy()

    if not stag50_chi_wins.empty:
        stag50_chi_wins['chi_diff'] = (
            stag50_chi_wins['mean_chi_Stag20'] - stag50_chi_wins['mean_chi_Stag50']
        ).round(4)
        stag50_chi_wins = stag50_chi_wins.sort_values(by='chi_diff', ascending=False)

        print("\n" + "="*80)
        print(f"{f'STAG 50 QUALITY WINS ({analysis_type}): DETAILED MARGINS':^80}")
        print("="*80)
        cols = ['instancia', 'mean_chi_Stag20', 'mean_chi_Stag50', 'chi_diff']
        print(stag50_chi_wins[cols].to_string(index=False))
        print("-" * 80)
        print(f"Average Improvement: {stag50_chi_wins['chi_diff'].mean():.4f} colors")
        print("="*80)
    else:
        print(f"\n[!] No quality wins found for Stag 50 in {analysis_type}.")

# Print margins for Pure
check_stag50_margins(df_pure_pivot, "PURE BRKGA")

# Print margins for Mutation
check_stag50_margins(df_mut_pivot, "MUTATION BRKGA")


                    RESULTS: PURE BRKGA                     
STRICTLY BY CHI:
chi_winner
Tied       27
Stag 20    17
Stag 50    10

STRICTLY BY TIME:
time_winner
Stag 20    54

ABSOLUTE WINNER:
abs_winner
Stag 20    44
Stag 50    10


                RESULTS: BRKGA WITH MUTATION                
STRICTLY BY CHI:
chi_winner
Tied       27
Stag 50    21
Stag 20     6

STRICTLY BY TIME:
time_winner
Stag 20    54

ABSOLUTE WINNER:
abs_winner
Stag 20    33
Stag 50    21


              STAG 50 QUALITY WINS (PURE BRKGA): DETAILED MARGINS               
                instancia  mean_chi_Stag20  mean_chi_Stag50  chi_diff
 simples_n500_p01%_v2.col            111.2            105.0       6.2
simples_n1500_p01%_v1.col            256.8            254.6       2.2
 simples_n500_p10%_v2.col            393.0            391.2       1.8
simples_n1500_p03%_v1.col            598.4            596.8       1.6
 simples_n500_p03%_v1.col            140.8            139.4       1.4
 simples_n100_p01%_v1.col   

**CONCLUSION**
For the pure BRKGA the result is very clear. stag = 50 barely has any wins in terms of chi and stag = 20 is consistently better, both in terms of chi and in terms of time. We will use the pure BRKGA with stag = 20.

For the BRKGAmutation variant, the result is not so clear. stag = 50 wins in chi a fair amount of times, but loses in time for every instance. As it didn't get an absolutely better performance and the mean difference is not so big, we will prefer to use BRKGA mutation with stag = 20 as well.

We will now continue to analysing which one, the pure BRKGA or BRKGAmutation with the already defined stag values, is going to perform better. We choose stag = 20 for both methods.

In [6]:
# 3B. tomada de decisão sobre qual é o melhor dos BRKGA's (grafos simples)
import pandas as pd

# logic for comparing methods with stag value already determined
def get_method_stats(row):
    # labels are now pure and mutation
    required_cols = ['mean_chi_Pure', 'mean_chi_Mutation', 'mean_time_Pure', 'mean_time_Mutation']
    if any(pd.isna(row.get(col)) for col in required_cols):
        return 'Incomplete', 'Incomplete', 'Incomplete'
    
    if row['mean_chi_Mutation'] < row['mean_chi_Pure']:
        chi_winner = 'Mutation'
    elif row['mean_chi_Pure'] < row['mean_chi_Mutation']:
        chi_winner = 'Pure'
    else:
        chi_winner = 'Tied'

    if row['mean_time_Mutation'] < row['mean_time_Pure']:
        time_winner = 'Mutation'
    elif row['mean_time_Pure'] < row['mean_time_Mutation']:
        time_winner = 'Pure'
    else:
        time_winner = 'Tied'

    abs_winner = chi_winner if chi_winner != 'Tied' else time_winner
    
    return chi_winner, time_winner, abs_winner

# comparison files (locked at stag 20)
comparison_files = {
    "Pure": "results_BRKGA_combined_stag20.csv",
    "Mutation": "results_BRKGAmutation_combined_stag20.csv"
}

# load and processing
all_results = []
for label, path in comparison_files.items():
    df = pd.read_csv(path)
    df_f = df[['instancia', 'mean_chi', 'mean_time']].copy()
    df_f['method_id'] = label
    all_results.append(df_f)

df_comp = pd.concat(all_results, ignore_index=True)
df_pivot = df_comp.pivot(index='instancia', columns='method_id', values=['mean_chi', 'mean_time'])
df_pivot.columns = [f'{col}_{val}'.replace(' ', '') for col, val in df_pivot.columns]
df_pivot = df_pivot.reset_index()

# check winners
df_pivot[['chi_winner', 'time_winner', 'abs_winner']] = df_pivot.apply(
    lambda r: pd.Series(get_method_stats(r)), axis=1
)

# result printing
valid = df_pivot[df_pivot['chi_winner'] != 'Incomplete']

print("\n" + "="*60)
print(f"{'METHOD COMPARISON: PURE vs MUTATION (FIXED STAG=20)':^60}")
print("="*60)
print(f"STRICTLY BY CHI (Quality):\n{valid['chi_winner'].value_counts().to_string()}\n")
print(f"STRICTLY BY TIME (Speed):\n{valid['time_winner'].value_counts().to_string()}\n")
print(f"ABSOLUTE WINNER (Chi > Time):\n{valid['abs_winner'].value_counts().to_string()}")
print("="*60 + "\n")

mut_quality_wins = valid[valid['chi_winner'] == 'Mutation'].copy()

if not mut_quality_wins.empty:
    # 2. Calculate Quality Improvement
    mut_quality_wins['chi_diff'] = (
        mut_quality_wins['mean_chi_Pure'] - mut_quality_wins['mean_chi_Mutation']
    ).round(4)

    # 3. Calculate Time Penalty (How much longer it took)
    mut_quality_wins['extra_time_sec'] = (
        mut_quality_wins['mean_time_Mutation'] - mut_quality_wins['mean_time_Pure']
    ).round(4)
    
    # Calculate % increase in time
    mut_quality_wins['time_increase_pct'] = (
        (mut_quality_wins['extra_time_sec'] / mut_quality_wins['mean_time_Pure']) * 100
    ).round(2)

    # Sort by the biggest quality improvement
    mut_quality_wins = mut_quality_wins.sort_values(by='chi_diff', ascending=False)

    print("\n" + "="*95)
    print(f"{'MUTATION QUALITY WINS: QUALITY GAIN vs TIME COST':^95}")
    print("="*95)
    
    # Display the metrics
    cols = ['instancia', 'chi_diff', 'mean_time_Pure', 'mean_time_Mutation', 'extra_time_sec', 'time_increase_pct']
    # Renaming for cleaner display
    display_df = mut_quality_wins[cols].rename(columns={
        'chi_diff': 'Chi Gain',
        'extra_time_sec': '+Time (s)',
        'time_increase_pct': '+Time (%)'
    })
    
    print(display_df.to_string(index=False))
    
    print("-" * 95)
    avg_extra = mut_quality_wins['extra_time_sec'].mean()
    avg_pct = mut_quality_wins['time_increase_pct'].mean()
    print(f"On average, Mutation found better colors but took {avg_extra:.2f}s longer ({avg_pct:.2f}% increase).")
    print("="*95)
else:
    print("\n[!] No quality wins found for Mutation vs Pure at Stag 20.")


    METHOD COMPARISON: PURE vs MUTATION (FIXED STAG=20)     
STRICTLY BY CHI (Quality):
chi_winner
Mutation    34
Tied        29
Pure         1

STRICTLY BY TIME (Speed):
time_winner
Pure        56
Mutation     8

ABSOLUTE WINNER (Chi > Time):
abs_winner
Mutation    42
Pure        22


                       MUTATION QUALITY WINS: QUALITY GAIN vs TIME COST                        
                instancia  Chi Gain  mean_time_Pure  mean_time_Mutation  +Time (s)  +Time (%)
simples_n2000_p05%_v1.col      47.8      390.760675         2467.121689  2076.3610     531.36
simples_n2000_p05%_v2.col      46.6      441.974259         2408.101614  1966.1274     444.85
simples_n1500_p05%_v1.col      38.6      120.561250          890.272117   769.7109     638.44
simples_n1500_p05%_v2.col      32.4      118.180962          806.381339   688.2004     582.33
 simples_n500_p01%_v2.col      31.0        5.421642           12.847920     7.4263     136.98
 simples_n500_p01%_v1.col      25.4        7.653048 

**CONCLUSION**
The trade-off is more explicit for comparing the different methods using the maximizaing stag value for each of them. BRKGAmutation gets big imporvements (up to 47.8 in color improvement) but also takes more time for it. As the color improvement is considerable, we take the BRKGAmutation method with stag = 20 as our final method. 

**1.2 - TESTES COM GENÉTICOS**

In [7]:
# 1° passo: preparação e tratamento dos arquivos que sejam necessários (merge, append, etc)

# merge para método gen com stag = 20
df_gen_p05_20 = pd.read_csv('results_GA_pmutation0.5_stag20_progresso.csv')
df_gen_n500_p05_20 = pd.read_csv('results_GA_n500_pmutation0.5_stag20_progresso.csv')

df_gen_p50_stag20_full = pd.concat([df_gen_p05_20, df_gen_n500_p05_20], ignore_index=True)
df_gen_p50_stag20_full.to_csv('results_GA_combined_pmutation0.5_stag20.csv', index=False)

# merge para método gen Binomial com stag = 50
df_genBinomial_p05_50 = pd.read_csv('results_GABinomial_pmutation0.5_stag50_progresso.csv')
df_genBinomial_n500_p05_50 = pd.read_csv('results_GABinomial_n500_pmutation0.5_stag50_progresso.csv')

df_genBinomial_p50_stag50_full = pd.concat([df_genBinomial_p05_50, df_genBinomial_n500_p05_50], ignore_index=True)
df_genBinomial_p50_stag50_full.to_csv('results_GABinomial_combined_pmutation0.5_stag50.csv', index=False)

# merge para método de gen Binomial com stag = 20
df_genBinomial_p05_20 = pd.read_csv('results_GABinomial_pmutation0.5_stag20_progresso.csv')
df_genBinomial_n500_p05_20 = pd.read_csv('results_GABinomial_n500_pmutation0.5_stag20_progresso.csv')

df_genBinomial_p50_stag20_full = pd.concat([df_genBinomial_p05_20, df_genBinomial_n500_p05_20], ignore_index=True)
df_genBinomial_p50_stag20_full.to_csv('results_GABinomial_combined_pmutation0.5_stag20.csv', index=False)


In [13]:
import pandas as pd
import numpy as np

# --- 1. CORE LOGIC ---
def get_detailed_stats(row):
    required_cols = ['mean_chi_Stag20', 'mean_chi_Stag50', 'mean_time_Stag20', 'mean_time_Stag50']
    if any(pd.isna(row.get(col)) for col in required_cols):
        return 'Incomplete', 'Incomplete', 'Incomplete'
    
    # Quality (Chi) - Lower is better
    if row['mean_chi_Stag20'] < row['mean_chi_Stag50']:
        chi_winner = 'Stag 20'
    elif row['mean_chi_Stag50'] < row['mean_chi_Stag20']:
        chi_winner = 'Stag 50'
    else:
        chi_winner = 'Tied'

    # Speed (Time) - Lower is better
    if row['mean_time_Stag20'] < row['mean_time_Stag50']:
        time_winner = 'Stag 20'
    elif row['mean_time_Stag50'] < row['mean_time_Stag20']:
        time_winner = 'Stag 50'
    else:
        time_winner = 'Tied'

    # Absolute Winner
    abs_winner = chi_winner if chi_winner != 'Tied' else time_winner 
    return chi_winner, time_winner, abs_winner

# --- 2. MASTER ANALYSIS FUNCTION ---
def run_full_analysis(files_dict, title):
    all_results = []
    cols_to_keep = ['instancia', 'mean_chi', 'mean_time']
    
    for label, path in files_dict.items():
        try:
            df = pd.read_csv(path)
            df.columns = df.columns.str.strip()
            df_f = df[[c for c in cols_to_keep if c in df.columns]].copy()
            df_f['parameter_id'] = label
            all_results.append(df_f)
        except FileNotFoundError:
            print(f"Skipping: {path} not found.")
            return None

    # Combine and Pivot (Using pivot_table to handle duplicates safely)
    df_combined = pd.concat(all_results, ignore_index=True)
    df_pivot = df_combined.pivot_table(
        index='instancia', 
        columns='parameter_id', 
        values=['mean_chi', 'mean_time'],
        aggfunc='mean'
    )
    df_pivot.columns = [f'{col}_{val}'.replace(' ', '') for col, val in df_pivot.columns]
    df_pivot = df_pivot.reset_index()

    # Determine Winners
    df_pivot[['chi_winner', 'time_winner', 'abs_winner']] = df_pivot.apply(
        lambda r: pd.Series(get_detailed_stats(r)), axis=1
    )

    valid = df_pivot[df_pivot['chi_winner'] != 'Incomplete'].copy()
    
    # --- PRINT SUMMARY ---
    print("\n" + "="*60)
    print(f"{title:^60}")
    print("="*60)
    if not valid.empty:
        print(f"STRICTLY BY CHI (Quality):\n{valid['chi_winner'].value_counts().to_string()}\n")
        print(f"ABSOLUTE WINNER:\n{valid['abs_winner'].value_counts().to_string()}")
        
        # --- DETAILED ANALYSIS: WHEN STAG 50 BEATS STAG 20 ---
        # (Checking if the extra computation time is worth the better Chi)
        stag50_quality_wins = valid[valid['chi_winner'] == 'Stag 50'].copy()
        
        if not stag50_quality_wins.empty:
            # Metrics
            stag50_quality_wins['chi_gain'] = (stag50_quality_wins['mean_chi_Stag20'] - stag50_quality_wins['mean_chi_Stag50']).round(4)
            stag50_quality_wins['extra_time'] = (stag50_quality_wins['mean_time_Stag50'] - stag50_quality_wins['mean_time_Stag20']).round(4)
            stag50_quality_wins['time_inc_%'] = ((stag50_quality_wins['extra_time'] / stag50_quality_wins['mean_time_Stag20']) * 100).round(2)
            
            print("\n" + "-"*85)
            print(f"{'STAG 50 QUALITY WINS: IS THE EXTRA TIME WORTH IT?':^85}")
            print("-"*85)
            display_cols = ['instancia', 'chi_gain', 'mean_time_Stag20', 'mean_time_Stag50', 'extra_time', 'time_inc_%']
            print(stag50_quality_wins[display_cols].sort_values('chi_gain', ascending=False).to_string(index=False))
            
            avg_gain = stag50_quality_wins['chi_gain'].mean()
            avg_time = stag50_quality_wins['time_inc_%'].mean()
            print(f"\nAverage Quality Gain: {avg_gain:.4f} | Average Time Increase: {avg_time:.2f}%")
        else:
            print("\n[!] Stag 50 never produced better quality than Stag 20 for this group.")
            
    else:
        print("No valid data for comparison.")
    print("="*85 + "\n")
    
    return df_pivot

# --- 3. EXECUTION ---

experiments = {
    "GA (Mutation 20%)": {
        "Stag 20": "results_GA_pmutation0.2_stag20_progresso.csv",
        "Stag 50": "results_GA_pmutation0.2_stag50_progresso.csv"
    },
    "GABinomial (Mutation 20%)": {
        "Stag 20": "results_GABinomial_pmutation0.2_stag20_progresso.csv",
        "Stag 50": "results_GABinomial_pmutation0.2_stag50_progresso.csv"
    },
    "GA (Mutation 50%)": {
        "Stag 20": "results_GA_combined_pmutation0.5_stag20.csv",
        "Stag 50": "results_GA_pmutation0.5_stag50_progresso.csv"
    },
    "GABinomial (Mutation 50%)": {
        "Stag 20": "results_GABinomial_combined_pmutation0.5_stag20.csv",
        "Stag 50": "results_GABinomial_combined_pmutation0.5_stag50.csv"
    },
    "GA (Mutation 80%)": {
        "Stag 20": "results_GA_pmutation0.8_stag20_progresso.csv",
        "Stag 50": "results_GA_pmutation0.8_stag50_progresso.csv"
    },
    "GABinomial (Mutation 80%)": {
        "Stag 20": "results_GABinomial_pmutation0.8_stag20_progresso.csv",
        "Stag 50": "results_GABinomial_pmutation0.8_stag50_progresso.csv"
    }
}

all_experiment_results = {}
for exp_name, file_set in experiments.items():
    res = run_full_analysis(file_set, exp_name)
    if res is not None:
        all_experiment_results[exp_name] = res


                     GA (Mutation 20%)                      
STRICTLY BY CHI (Quality):
chi_winner
Tied       24
Stag 50    20
Stag 20     4

ABSOLUTE WINNER:
abs_winner
Stag 20    28
Stag 50    20

-------------------------------------------------------------------------------------
                  STAG 50 QUALITY WINS: IS THE EXTRA TIME WORTH IT?                  
-------------------------------------------------------------------------------------
                instancia  chi_gain  mean_time_Stag20  mean_time_Stag50  extra_time  time_inc_%
simples_n1500_p05%_v1.col      11.6        203.531801       1150.619647    947.0878      465.33
 simples_n500_p01%_v2.col      10.0          9.181077         14.945011      5.7639       62.78
simples_n1500_p05%_v2.col       7.0        212.875308        778.474952    565.5996      265.70
simples_n1000_p01%_v2.col       6.2         28.698277         56.843543     28.1453       98.07
 simples_n500_p10%_v2.col       5.8         27.874784         

**CONCLUSION**
In this case, the relationship between the metrics and the results are more nuanced. The stag = 20 value consistently gets the better time, but also consistently gets beaten/equaled to the stag = 50 value. We will discuss that and come back later. For the development of the code, we will just use stag = 20 for all methods.

In [19]:
import pandas as pd
import numpy as np

def run_comprehensive_triple_analysis(algorithm_name, files_dict):
    all_results = []
    
    # 1. Load Data
    for prob_label, path in files_dict.items():
        try:
            df = pd.read_csv(path)
            df.columns = df.columns.str.strip()
            df_f = df[['instancia', 'mean_chi', 'mean_time']].copy()
            df_f['prob'] = prob_label
            all_results.append(df_f)
        except FileNotFoundError:
            print(f"Skipping: {path} not found.")
            continue

    if not all_results: return None

    # 2. Pivot Table (Handling duplicates with mean)
    df_combined = pd.concat(all_results, ignore_index=True)
    df_pivot = df_combined.pivot_table(
        index='instancia', 
        columns='prob', 
        values=['mean_chi', 'mean_time'],
        aggfunc='mean'
    )
    df_pivot.columns = [f'{col}_{val}' for col, val in df_pivot.columns]
    df_pivot = df_pivot.reset_index()

    # 3. Define Winners Logic
    chi_cols = [c for c in df_pivot.columns if 'mean_chi' in c]
    time_cols = [c for c in df_pivot.columns if 'mean_time' in c]
    probs = list(files_dict.keys())

    def get_winners(row):
        # Quality (Chi)
        best_chi = row[chi_cols].min()
        chi_wins = [c.split('_')[-1] for c in chi_cols if row[c] == best_chi]
        chi_win = chi_wins[0] if len(chi_wins) == 1 else "Tie"

        # Speed (Time)
        best_time = row[time_cols].min()
        time_wins = [c.split('_')[-1] for c in time_cols if row[c] == best_time]
        time_win = time_wins[0] if len(time_wins) == 1 else "Tie"

        # Absolute (Chi priority)
        overall_win = chi_win if chi_win != "Tie" else time_win
        return pd.Series([chi_win, time_win, overall_win])

    df_pivot[['Chi_Winner', 'Time_Winner', 'Overall_Winner']] = df_pivot.apply(get_winners, axis=1)

    # --- PRINTING SECTION ---
    print("\n" + "="*100)
    print(f"PERFORMANCE REPORT: {algorithm_name} ".center(100, "="))
    print("="*100)

    # PRINT 1: Absolute Winner Counts
    print("\n[SUMMARY: TOTAL WINS BY PROBABILITY]")
    counts = pd.DataFrame({
        'Quality (Chi)': df_pivot['Chi_Winner'].value_counts(),
        'Speed (Time)': df_pivot['Time_Winner'].value_counts(),
        'Absolute Winner': df_pivot['Overall_Winner'].value_counts()
    }).fillna(0).astype(int)
    print(counts)

    # PRINT 2: 2-by-2 Head-to-Head (Round Robin)
    print("\n" + "-"*40)
    print(" HEAD-TO-HEAD QUALITY (CHI) BATTLES ".center(40, "-"))
    for i in range(len(probs)):
        for j in range(i + 1, len(probs)):
            p1, p2 = probs[i], probs[j]
            c1, c2 = f'mean_chi_{p1}', f'mean_chi_{p2}'
            p1_wins = (df_pivot[c1] < df_pivot[c2]).sum()
            p2_wins = (df_pivot[c2] < df_pivot[c1]).sum()
            ties = (df_pivot[c1] == df_pivot[c2]).sum()
            print(f" > {p1:^4} vs {p2:^4} | {p1} wins: {p1_wins} | {p2} wins: {p2_wins} | Ties: {ties}")

    # PRINT 3: Quality vs. Time Overhead (Baseline 20%)
    print("\n" + "-"*100)
    print(" QUALITY GAIN VS TIME PENALTY (Relative to 20% Baseline) ".center(100, "-"))
    for target in ["50%", "80%"]:
        sub = df_pivot[df_pivot[f'mean_chi_{target}'] < df_pivot['mean_chi_20%']].copy()
        if not sub.empty:
            sub['chi_gain'] = (sub['mean_chi_20%'] - sub[f'mean_chi_{target}']).round(4)
            sub['time_inc_%'] = (((sub[f'mean_time_{target}'] - sub['mean_time_20%']) / sub['mean_time_20%']) * 100).round(2)
            print(f"\n>>> Instances where {target} found BETTER quality than 20%:")
            print(sub[['instancia', 'chi_gain', 'time_inc_%']].sort_values('chi_gain', ascending=False).to_string(index=False))
            print(f"Average {target} Gain: {sub['chi_gain'].mean():.4f} | Avg Time Increase: {sub['time_inc_%'].mean():.2f}%")

    # PRINT 4: High-Mutation Battle (80% vs 50%)
    print("\n" + "-"*100)
    print(" BATTLE OF THE HIGHS: 80% vs 50% OVERHEAD ".center(100, "-"))
    
    # 80% beats 50%
    wins_80 = df_pivot[df_pivot['mean_chi_80%'] < df_pivot['mean_chi_50%']].copy()
    if not wins_80.empty:
        wins_80['chi_gain'] = (wins_80['mean_chi_50%'] - wins_80['mean_chi_80%']).round(4)
        wins_80['time_inc_%'] = (((wins_80['mean_time_80%'] - wins_80['mean_time_50%']) / wins_80['mean_time_50%']) * 100).round(2)
        print(f"\n>>> 80% found BETTER quality than 50%:")
        print(wins_80[['instancia', 'chi_gain', 'time_inc_%']].to_string(index=False))
    else:
        print("\n>>> 80% mutation never produced better results than 50% in any instance.")

    # 50% beats 80% (Efficiency check)
    wins_50 = df_pivot[df_pivot['mean_chi_50%'] < df_pivot['mean_chi_80%']].copy()
    if not wins_50.empty:
        print(f"\n>>> 50% was BETTER than 80% in quality ({len(wins_50)} instances):")
        wins_50['time_saved_%'] = (((wins_50['mean_time_80%'] - wins_50['mean_time_50%']) / wins_50['mean_time_80%']) * 100).round(2)
        print(wins_50[['instancia', 'time_saved_%']].to_string(index=False))

    print("="*100 + "\n")
    return df_pivot

# --- EXECUTION ---
for stag in ["stag20", "stag50"]:
    # GA
    ga_files = {
        "20%": f"results_GA_pmutation0.2_{stag}_progresso.csv",
        "50%": f"results_GA_combined_pmutation0.5_{stag}.csv" if stag == "stag20" else f"results_GA_pmutation0.5_{stag}_progresso.csv",
        "80%": f"results_GA_pmutation0.8_{stag}_progresso.csv"
    }
    run_comprehensive_triple_analysis(f"GA ({stag})", ga_files)

    # GABinomial
    gab_files = {
        "20%": f"results_GABinomial_pmutation0.2_{stag}_progresso.csv",
        "50%": f"results_GABinomial_combined_pmutation0.5_{stag}.csv",
        "80%": f"results_GABinomial_pmutation0.8_{stag}_progresso.csv"
    }
    run_comprehensive_triple_analysis(f"GABinomial ({stag})", gab_files)


==================================PERFORMANCE REPORT: GA (stag20) ==================================

[SUMMARY: TOTAL WINS BY PROBABILITY]
     Quality (Chi)  Speed (Time)  Absolute Winner
20%              5            10               10
50%             14            48               39
80%             12             6               15
Tie             33             0                0

----------------------------------------
-- HEAD-TO-HEAD QUALITY (CHI) BATTLES --
 > 20%  vs 50%  | 20% wins: 7 | 50% wins: 20 | Ties: 21
 > 20%  vs 80%  | 20% wins: 7 | 80% wins: 19 | Ties: 22
 > 50%  vs 80%  | 50% wins: 9 | 80% wins: 14 | Ties: 33

----------------------------------------------------------------------------------------------------
--------------------- QUALITY GAIN VS TIME PENALTY (Relative to 20% Baseline) ----------------------

>>> Instances where 50% found BETTER quality than 20%:
                instancia  chi_gain  time_inc_%
simples_n1000_p01%_v1.col       8.4       -0.85
simp

In [22]:
# 3.1.S. GA e GABinomial comparação entre métodos (com parâmetros que maximizam performance já definidos)
import pandas as pd
import numpy as np

def run_final_method_comparison(stag_label, ga_path, gab_path):
    """
    Final comparison between GA and GABinomial for a fixed stag and 
    the best identified mutation probability.
    """
    all_results = []
    files = {"GA": ga_path, "GABinomial": gab_path}
    
    # 1. Load and Label
    for label, path in files.items():
        try:
            df = pd.read_csv(path)
            df.columns = df.columns.str.strip()
            df_f = df[['instancia', 'mean_chi', 'mean_time']].copy()
            df_f['method'] = label
            all_results.append(df_f)
        except FileNotFoundError:
            print(f"Skipping: {path} not found.")
            return None

    # 2. Pivot
    df_combined = pd.concat(all_results, ignore_index=True)
    df_pivot = df_combined.pivot_table(
        index='instancia', 
        columns='method', 
        values=['mean_chi', 'mean_time'],
        aggfunc='mean'
    )
    df_pivot.columns = [f'{col}_{val}' for col, val in df_pivot.columns]
    df_pivot = df_pivot.reset_index()

    # 3. Winner Logic
    def get_final_winner(row):
        # Quality
        if row['mean_chi_GABinomial'] < row['mean_chi_GA']:
            chi_win = "GABinomial"
        elif row['mean_chi_GA'] < row['mean_chi_GABinomial']:
            chi_win = "GA"
        else:
            chi_win = "Tie"

        # Speed
        time_win = "GABinomial" if row['mean_time_GABinomial'] < row['mean_time_GA'] else "GA"
        
        # Absolute (Quality > Speed)
        overall = chi_win if chi_win != "Tie" else time_win
        return pd.Series([chi_win, time_win, overall])

    df_pivot[['Chi_Winner', 'Time_Winner', 'Overall_Winner']] = df_pivot.apply(get_final_winner, axis=1)

    # --- PRINTING ---
    print("\n" + "="*100)
    print(f" FINAL METHOD SELECTION: GA vs GABinomial ({stag_label}) ".center(100, "="))
    print("="*100)

    # Summary Counts
    print("\n[SUMMARY OF WINS]")
    summary = pd.DataFrame({
        'Quality (Chi)': df_pivot['Chi_Winner'].value_counts(),
        'Speed (Time)': df_pivot['Time_Winner'].value_counts(),
        'Absolute Winner': df_pivot['Overall_Winner'].value_counts()
    }).fillna(0).astype(int)
    print(summary)

    # Quality Wins Analysis (Does Binomial pay off?)
    gab_wins = df_pivot[df_pivot['Chi_Winner'] == 'GABinomial'].copy()
    print("\n" + "-"*100)
    print(" GABINOMIAL QUALITY ADVANTAGE vs GA OVERHEAD ".center(100, "-"))
    
    if not gab_wins.empty:
        gab_wins['chi_gain'] = (gab_wins['mean_chi_GA'] - gab_wins['mean_chi_GABinomial']).round(4)
        gab_wins['time_diff_sec'] = (gab_wins['mean_time_GABinomial'] - gab_wins['mean_time_GA']).round(4)
        gab_wins['time_inc_%'] = ((gab_wins['time_diff_sec'] / gab_wins['mean_time_GA']) * 100).round(2)
        
        print(gab_wins[['instancia', 'chi_gain', 'time_diff_sec', 'time_inc_%']].sort_values('chi_gain', ascending=False).to_string(index=False))
        print(f"\nAverage Chi Gain: {gab_wins['chi_gain'].mean():.4f} | Avg Time Penalty: {gab_wins['time_inc_%'].mean():.2f}%")
    else:
        print("GABinomial did not achieve better quality than GA in any instance for this configuration.")

    # GA Efficiency Check (Where GA is just better/faster)
    ga_wins = df_pivot[df_pivot['Chi_Winner'] == 'GA'].copy()
    if not ga_wins.empty:
        print(f"\n>>> GA was SUPERIOR in Quality in {len(ga_wins)} instances.")
        
    print("="*100 + "\n")
    return df_pivot

best_prob = "0.5"

# stag 20
run_final_method_comparison(
    "Stag 20 / Prob 50%",
    ga_path = f"results_GA_combined_pmutation{best_prob}_stag20.csv",
    gab_path = f"results_GABinomial_combined_pmutation{best_prob}_stag20.csv"
)

# stag 50
run_final_method_comparison(
    "Stag 50 / Prob 50%",
    ga_path = f"results_GA_pmutation{best_prob}_stag50_progresso.csv",
    gab_path = f"results_GABinomial_combined_pmutation{best_prob}_stag50.csv"
)


================== FINAL METHOD SELECTION: GA vs GABinomial (Stag 20 / Prob 50%) ===================

[SUMMARY OF WINS]
            Quality (Chi)  Speed (Time)  Absolute Winner
GA                      4            63               40
GABinomial             24             1               24
Tie                    36             0                0

----------------------------------------------------------------------------------------------------
--------------------------- GABINOMIAL QUALITY ADVANTAGE vs GA OVERHEAD ----------------------------
                instancia  chi_gain  time_diff_sec  time_inc_%
simples_n1500_p05%_v1.col      22.4       582.6175      199.75
simples_n1500_p05%_v2.col      21.0       785.0332      677.11
 simples_n500_p01%_v1.col      19.0         7.8620       66.98
 simples_n500_p01%_v2.col      14.4        10.2120      119.41
simples_n1000_p05%_v1.col      11.6       130.1140      283.29
simples_n1000_p05%_v2.col      11.2       128.1372      293.23
 simple

,instancia,mean_chi_GA,mean_chi_GABinomial,mean_time_GA,mean_time_GABinomial,Chi_Winner,Time_Winner,Overall_Winner
0,simples_n1000_p01%_v1.col,162.6,163.3,54.448117,63.409072,GA,GA,GA
1,simples_n1000_p01%_v2.col,164.0,162.3,58.856296,55.480572,GABinomial,GABinomial,GABinomial
2,simples_n1000_p03%_v1.col,343.4,343.0,119.518440,128.981571,GABinomial,GA,GABinomial
3,simples_n1000_p03%_v2.col,343.6,339.6,83.168556,117.216252,GABinomial,GA,GABinomial
4,simples_n1000_p05%_v1.col,526.4,521.0,218.865911,282.347915,GABinomial,GA,GABinomial
...,...,...,...,...,...,...,...,...
59,simples_n500_p20%_v2.col,500.0,500.0,80.081617,133.994821,Tie,GA,GA
60,simples_n500_p30%_v1.col,500.0,500.0,172.655119,285.238376,Tie,GA,GA
61,simples_n500_p30%_v2.col,500.0,500.0,173.922005,284.882815,Tie,GA,GA
62,simples_n500_p40%_v1.col,500.0,NaN,303.623530,NaN,Tie,GA,GA
